# Super Midsize Jet Premium Extraction Audit — Challenger Fleet

**Segment:** Super Midsize Jet (SMID) &nbsp;|&nbsp; **Fleet:** Challenger series &nbsp;|&nbsp; 
**Data pipeline:** Strategic Rhombus — `smid_final_wingx_24_25` reference notebook

---

## Purpose

This notebook identifies the **highest-value pricing windows** across 20 SMID strategic corridors.
The analysis mirrors the LJ methodology with three SMID-specific calibrations:

| Calibration | LJ value | SMID value | Reason |
|---|---|---|---|
| Velocity floor | 15% | **12%** | Denver-hub corridors distribute demand more evenly across TOD slots |
| National WUP baseline | 4.0% | **3.4%** | Segment-specific overall WUP SMID long-haul share (ref Cell 5 source notebook) |
| Stronghold WUP threshold | > 6.5% | **> 7.2%** | SMID distribution has higher top-end WUP concentration |

---

## Data Sources

| File | Records | Coverage |
|---|---|---|
| `smid_dow_heatmaps.json` | 140 | 20 corridors × 7 days-of-week |
| `smid_tod_heatmaps.json` | 120 | 20 corridors × 6 time-of-day buckets |

Pre-computed corridor aggregates from the Strategic Rhombus pipeline.
Raw flight-level data is processed upstream — this notebook ingests summary records only.

---

## Column Glossary

| Column | Definition |
|---|---|
| `tod_market_velocity_pct` | % of corridor's annual flights occurring in this TOD window — measures market depth |
| `tod_wup_share_pct` | WUP flights ÷ total flights in that window — the core extraction signal |
| `dow_market_velocity_pct` | % of corridor's annual flights on this day-of-week |
| `dow_wup_share_pct` | WUP flights ÷ total flights on that day-of-week |

---

## Analysis Pipeline

| Cell | Output | Purpose |
|---|---|---|
| 1–3 | `df_dow_final`, `df_tod_final` | Load and normalize raw heatmap records |
| 4 | `df_best` (20 rows) | Rank-sum selection — one best DOW day + one best TOD slot per corridor |
| 5 | `df_categorized` (20 rows) | Premium extraction classification into 4 tiers |
| 6 | `df_value` (20 rows) | Revenue opportunity quantification per corridor |
| 7–8 | `df_signals` | Active corridors only (No Signal removed) |


In [1]:
# -----------------------------------------------------------------------------------------
# CELL 1: LOAD PRE-COMPUTED SMID HEATMAP DATA
# Source: smid_dow_heatmaps.json  — 140 records (20 corridors x 7 DOW days)
#         smid_tod_heatmaps.json  — 120 records (20 corridors x 6 TOD slots)
# Derived from the Strategic Rhombus pipeline.
# -----------------------------------------------------------------------------------------
import pandas as pd
import numpy as np

df_dow = pd.read_json("smid_dow_heatmaps.json")
df_tod = pd.read_json("smid_tod_heatmaps.json")

print("Loading SMID heatmap data...")
print(f"  DOW records : {len(df_dow['records']):,}")
print(f"  TOD records : {len(df_tod['records']):,}")


Loading SMID heatmap data...
  DOW records : 140
  TOD records : 120


In [2]:
# -----------------------------------------------------------------------------------------
# CELL 2: NORMALIZE DOW RECORDS
# Output: df_dow_final — 140 rows
#         columns: corridor | dimension | dow_market_velocity_pct | dow_wup_share_pct
# -----------------------------------------------------------------------------------------
df_dow_final = pd.json_normalize(df_dow["records"])

print(f"DOW shape : {df_dow_final.shape}")
print(f"Corridors : {df_dow_final['corridor'].nunique()}")
print(f"Days      : {sorted(df_dow_final['dimension'].unique())}")
display(df_dow_final.head(7))


DOW shape : (140, 4)
Corridors : 20
Days      : ['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']


,corridor,dimension,dow_market_velocity_pct,dow_wup_share_pct
0,New York City ➔ South Florida,Monday,13.7199,1.6290
1,New York City ➔ South Florida,Tuesday,12.0561,0.8239
2,New York City ➔ South Florida,Wednesday,15.3092,1.0543
3,New York City ➔ South Florida,Thursday,17.3578,1.0014
4,New York City ➔ South Florida,Friday,16.5632,1.1244
5,New York City ➔ South Florida,Saturday,10.2930,1.4475
6,New York City ➔ South Florida,Sunday,14.7008,1.4358


In [3]:
# -----------------------------------------------------------------------------------------
# CELL 3: NORMALIZE TOD RECORDS
# Output: df_tod_final — 120 rows
#         columns: corridor | dimension | tod_market_velocity_pct | tod_wup_share_pct
# -----------------------------------------------------------------------------------------
df_tod_final = pd.json_normalize(df_tod["records"])

print(f"TOD shape : {df_tod_final.shape}")
print(f"Corridors : {df_tod_final['corridor'].nunique()}")
print(f"Buckets   : {sorted(df_tod_final['dimension'].unique())}")
display(df_tod_final.head(6))


TOD shape : (120, 4)
Corridors : 20
Buckets   : ['07:00-09:59', '10:00-12:59', '13:00-15:59', '16:00-18:59', '19:00-21:59', 'Off-Peak (Late/Early)']


,corridor,dimension,tod_market_velocity_pct,tod_wup_share_pct
0,New York City ➔ South Florida,07:00-09:59,18.1276,0.7534
1,New York City ➔ South Florida,10:00-12:59,28.4579,1.3089
2,New York City ➔ South Florida,13:00-15:59,24.1122,1.3388
3,New York City ➔ South Florida,16:00-18:59,18.3760,1.6216
4,New York City ➔ South Florida,19:00-21:59,7.5490,0.4934
5,New York City ➔ South Florida,Off-Peak (Late/Early),3.3772,1.1029


## Data Structure After Normalization

Each JSON file contains one record per corridor-dimension pair.
Example TOD record:

    { "corridor": "South Florida ➔ Denver",
      "dimension": "10:00-12:59",
      "tod_market_velocity_pct": 42.54,
      "tod_wup_share_pct": 9.23 }

**SMID-specific data characteristic — Denver hub concentration:**

The SMID corridor set is Denver-hub heavy, which creates a distinct demand pattern:
- Denver-hub corridors often have **flatter velocity distributions** across TOD slots (no single dominant peak)
- This means the rank-sum "winner" may have a velocity of 14–15%, below the LJ floor of 15%
- A 14.6% velocity on Denver → Boston is still a genuine active window — the 12% floor is calibrated accordingly
- Denver → Boston (vel=14.6%, wup=8.61%) is correctly classified as **Stronghold**: WUP concentration
  is exceptional (8.61% = 2.53× national average) even if velocity is moderate


## Rank-Sum Dimension Selection — How It Works

For each corridor we pick **one DOW day** and **one TOD slot** where velocity and WUP share are simultaneously highest.
A simple "pick the max" approach fails because the fastest slot is rarely the one with the highest share.
Instead we use a **rank-sum**: rank each slot independently on both metrics, add the ranks, and pick the slot with the lowest total.

### Algorithm (applied separately to DOW and TOD)

1. For every slot in a corridor, assign `vel_rank` — rank **1 = fastest** (descending).
2. For every slot in a corridor, assign `wup_rank` — rank **1 = highest WUP share** (descending).
3. `combined_rank = vel_rank + wup_rank`
4. Pick the slot with the **lowest** `combined_rank`.
5. Tiebreaker (equal `combined_rank`): highest `wup_share`, then highest `velocity`.

A slot that ranks #1 on both gets `combined_rank = 2` — the theoretical minimum.

---

### Hand-Trace Example — South Florida → Denver (SMID's top corridor)

| Time Slot | tod_velocity | tod_wup | vel_rank | wup_rank | combined_rank |
|---|---|---|---|---|---|
| 07:00–09:59 | 14.2% | 3.1% | 5 | 3 | **8** |
| **10:00–12:59** | **42.5%** | **9.2%** | **1** | **1** | **2** ← winner |
| 13:00–15:59 | 21.8% | 5.7% | 3 | 2 | **5** |
| 16:00–18:59 | 31.4% | 2.4% | 2 | 4 | **6** |
| 19:00–21:59 | 12.1% | 1.8% | 4 | 5 | **9** |
| Off-Peak | 3.1% | 0.6% | 6 | 6 | **12** |

`10:00–12:59` ranks **#1 on velocity AND #1 on WUP share** → `combined_rank = 2` → selected.
This is the ideal outcome: both signals peak simultaneously.

---

### What the rank-sum protects against

| Scenario | Pure-velocity pick | Pure-WUP pick | Rank-sum pick |
|---|---|---|---|
| Fastest slot has near-zero WUP share | ✗ wrong slot | — | ✓ balanced |
| Best-WUP slot is off-peak (vel ≈ 2%) | — | ✗ wrong slot | ✓ balanced |
| Both metrics peak at the same slot | ✓ correct | ✓ correct | ✓ correct |
| Metrics peak at different slots | ✗ misses WUP | ✗ misses demand | ✓ finds best balance |

---

### SMID Edge Case — Denver → Boston (vel = 14.6%, wup = 8.61%)

Denver → Boston wins at the **16:00–18:59 slot** with velocity 14.6% — below the LJ floor of 15%.
This is intentional: the 12% floor was set specifically for Denver-hub corridors.
The WUP concentration of 8.61% (SLI = 2.53) is the second-strongest in the SMID set.
The 16:00–18:59 window is the best *balanced* slot — other slots have higher velocity but
significantly lower WUP share. Rank-sum correctly identifies this as the actionable window.

---

### SMID-Specific Parameters

| Parameter | Value | Source |
|---|---|---|
| Velocity floor | 12% | Denver-hub demand distributes more evenly across slots |
| National WUP baseline | 3.4% | `overall_wup_share = (total_wup_missions / total_long_haul) * 100` — ref notebook Cell 5 |
| SLI formula | `tod_wup_share_pct / 3.4` | Measures WUP's concentration vs SMID segment average |


In [4]:
# -----------------------------------------------------------------------------------------
# CELL 4: BEST DOW + TOD PER CORRIDOR — RANK-SUM SELECTION
# Logic  : vel_rank + wup_rank = combined_rank.  Lowest = closest to #1 on BOTH.
# Tiebreak: highest wup_share, then highest velocity.
# Output : df_best — 20 rows, one per corridor.
# -----------------------------------------------------------------------------------------

# ── DOW ─────────────────────────────────────────────────────────────────────
df_dow_ranked = df_dow_final.copy()
df_dow_ranked["vel_rank"] = (
    df_dow_ranked.groupby("corridor")["dow_market_velocity_pct"]
    .rank(ascending=False, method="min")
)
df_dow_ranked["wup_rank"] = (
    df_dow_ranked.groupby("corridor")["dow_wup_share_pct"]
    .rank(ascending=False, method="min")
)
df_dow_ranked["combined_rank"] = df_dow_ranked["vel_rank"] + df_dow_ranked["wup_rank"]

df_best_dow = (
    df_dow_ranked
    .sort_values(
        ["corridor", "combined_rank", "dow_wup_share_pct", "dow_market_velocity_pct"],
        ascending=[True, True, False, False]
    )
    .groupby("corridor", sort=True).first().reset_index()
    .rename(columns={
        "dimension":               "dow_dimension",
        "dow_market_velocity_pct": "dow_velocity_pct",
    })
    [["corridor", "dow_dimension", "dow_velocity_pct", "dow_wup_share_pct"]]
)

# ── TOD ─────────────────────────────────────────────────────────────────────
df_tod_ranked = df_tod_final.copy()
df_tod_ranked["vel_rank"] = (
    df_tod_ranked.groupby("corridor")["tod_market_velocity_pct"]
    .rank(ascending=False, method="min")
)
df_tod_ranked["wup_rank"] = (
    df_tod_ranked.groupby("corridor")["tod_wup_share_pct"]
    .rank(ascending=False, method="min")
)
df_tod_ranked["combined_rank"] = df_tod_ranked["vel_rank"] + df_tod_ranked["wup_rank"]

df_best_tod = (
    df_tod_ranked
    .sort_values(
        ["corridor", "combined_rank", "tod_wup_share_pct", "tod_market_velocity_pct"],
        ascending=[True, True, False, False]
    )
    .groupby("corridor", sort=True).first().reset_index()
    .rename(columns={
        "dimension":               "tod_dimension",
        "tod_market_velocity_pct": "tod_velocity_pct",
    })
    [["corridor", "tod_dimension", "tod_velocity_pct", "tod_wup_share_pct"]]
)

# ── Merge ────────────────────────────────────────────────────────────────────
df_best = (
    df_best_dow
    .merge(df_best_tod, on="corridor")
    [["corridor", "dow_dimension", "tod_dimension",
      "dow_velocity_pct", "tod_velocity_pct",
      "dow_wup_share_pct", "tod_wup_share_pct"]]
)

print(f"Best-dimension table: {df_best.shape}  (expect 20 x 7)")
display(df_best)


Best-dimension table: (20, 7)  (expect 20 x 7)


,corridor,dow_dimension,tod_dimension,dow_velocity_pct,tod_velocity_pct,dow_wup_share_pct,tod_wup_share_pct
0,Atlanta ➔ Denver,Thursday,13:00-15:59,16.5734,28.0420,7.5949,7.9800
1,Boston ➔ Denver,Sunday,07:00-09:59,17.6390,23.8161,9.3385,7.7810
2,Boston ➔ South Florida,Thursday,10:00-12:59,15.6514,32.2868,3.4431,4.6444
3,Chicago ➔ Denver,Thursday,16:00-18:59,16.3307,16.4858,1.8987,3.7618
4,Chicago ➔ South Florida,Thursday,10:00-12:59,16.2087,33.6348,2.3605,1.8614
5,DMV ➔ Denver,Sunday,13:00-15:59,12.8511,24.5787,9.2896,6.2857
6,Dallas ➔ New York City,Wednesday,07:00-09:59,15.8099,17.8952,3.0675,2.1680
7,Denver ➔ Boston,Monday,16:00-18:59,15.6162,14.6359,10.7623,8.6124
8,Denver ➔ New York City,Sunday,10:00-12:59,19.5061,38.6880,7.1611,6.7054
9,Denver ➔ South Florida,Saturday,10:00-12:59,16.0600,41.3888,13.1429,8.3518


## Step 3 — Premium Extraction Classification (SMID-Calibrated)

### Threshold derivation

All 20 SMID corridor TOD winners were sorted by `tod_wup_share_pct` descending.
Cut points placed at **natural distributional gaps**:

| Rank | Corridor | WUP% | Gap to next |
|---|---|---|---|
| 5 | Boston → Denver | 7.78% | ↓ **1.07 pp gap** |
| 6 | Denver → New York City | 6.71% | |
| 8 | DMV → Denver | 6.29% | ↓ **1.65 pp gap** |
| 9 | Boston → South Florida | 4.64% | |
| 13 | South Florida → Boston | 3.02% | ↓ **0.47 pp gap** |
| 14 | Minneapolis → South Florida | 2.55% | |

Cut points: **> 7.2%** (top tier), **≥ 4.0%** (second tier), **≥ 2.25%** (third tier).

### Category definitions (SMID)

| Category | Condition | Revenue action |
|---|---|---|
| **Stronghold Extraction** | wup > 7.2% AND velocity > 12% | Maximum premium +25–30% — dominant Denver-hub WUP position |
| **Surge Capture** | 4.0% ≤ wup ≤ 7.2% AND velocity > 12% | Aggressive push +15–20% — strong and active windows |
| **Niche Protection** | 2.25% ≤ wup < 4.0% | Moderate uplift +10–15% — defend share, no velocity gate needed |
| **Velocity Opportunity** | velocity > 25% AND 2.0% ≤ wup < 2.25% | Selective push — high demand, early-stage WUP presence |
| **No Signal** | all other | No pricing action |

### Dual-criterion gate (same as LJ)

Every active SMID candidate satisfies **both**:
- ✅ Strong market demand: velocity ≥ 12% (SMID-calibrated floor)
- ✅ Meaningful WUP presence: wup ≥ 2.0% (no sub-2% extractions)

Note: Velocity Opportunity has **0 qualifiers** in the SMID dataset — both candidates (Chicago → SF at 1.86%,
SF → New York City at 0.63%) fall below the 2.0% WUP floor and are correctly assigned to No Signal.


In [5]:
# -----------------------------------------------------------------------------------------
# CELL 5: PREMIUM EXTRACTION CATEGORIES (TOD-DRIVEN, SMID-CALIBRATED)
#
# Thresholds recalibrated from SMID rank-sum winner distribution:
#   Stronghold Extraction  (+30/25%)  tod_wup > 7.2%    AND tod_velocity > 12%
#   Surge Capture          (+20%)     tod_wup 4.0-7.2%  AND tod_velocity > 12%
#   Niche Protection       (+15/10%)  tod_wup 2.25-4.0%
#   Velocity Opportunity   (+20/15%)  tod_velocity > 25% AND tod_wup 2.0-2.25%
#   No Signal                         low velocity AND low wup
#
# Velocity floor 12% (vs LJ 15%) — Denver-hub corridors show lower peak-slot velocities.
# -----------------------------------------------------------------------------------------

conditions = [
    (df_best["tod_wup_share_pct"] > 7.2)   & (df_best["tod_velocity_pct"] > 12),
    (df_best["tod_wup_share_pct"] >= 4.0)  & (df_best["tod_wup_share_pct"] <= 7.2) & (df_best["tod_velocity_pct"] > 12),
    (df_best["tod_wup_share_pct"] >= 2.25) & (df_best["tod_wup_share_pct"] <  4.0),
    (df_best["tod_velocity_pct"]  > 25)    & (df_best["tod_wup_share_pct"] >= 2.0) & (df_best["tod_wup_share_pct"] <  2.25),
]
labels = ["Stronghold Extraction", "Surge Capture", "Niche Protection", "Velocity Opportunity"]

df_categorized = df_best.copy()
df_categorized["category"] = np.select(conditions, labels, default="No Signal")

df_categorized = df_categorized[[
    "corridor",
    "dow_dimension", "dow_velocity_pct", "dow_wup_share_pct",
    "tod_dimension", "tod_velocity_pct", "tod_wup_share_pct",
    "category",
]]

category_order = ["Stronghold Extraction", "Surge Capture", "Niche Protection", "Velocity Opportunity", "No Signal"]
df_categorized["category"] = pd.Categorical(df_categorized["category"], categories=category_order, ordered=True)
df_categorized = (
    df_categorized
    .sort_values(["category", "tod_velocity_pct", "tod_wup_share_pct"], ascending=[True, False, False])
    .reset_index(drop=True)
)

print("SMID PREMIUM EXTRACTION CATEGORY DISTRIBUTION")
print("-" * 48)
print(df_categorized["category"].value_counts().reindex(category_order).to_string())
print()
display(df_categorized)


SMID PREMIUM EXTRACTION CATEGORY DISTRIBUTION
------------------------------------------------
category
Stronghold Extraction    5
Surge Capture            5
Niche Protection         5
Velocity Opportunity     0
No Signal                5



,corridor,dow_dimension,dow_velocity_pct,dow_wup_share_pct,tod_dimension,tod_velocity_pct,tod_wup_share_pct,category
0,South Florida ➔ Denver,Friday,14.7040,10.5033,10:00-12:59,42.5354,9.2284,Stronghold Extraction
1,Denver ➔ South Florida,Saturday,16.0600,13.1429,10:00-12:59,41.3888,8.3518,Stronghold Extraction
2,Atlanta ➔ Denver,Thursday,16.5734,7.5949,13:00-15:59,28.0420,7.9800,Stronghold Extraction
3,Boston ➔ Denver,Sunday,17.6390,9.3385,07:00-09:59,23.8161,7.7810,Stronghold Extraction
4,Denver ➔ Boston,Monday,15.6162,10.7623,16:00-18:59,14.6359,8.6124,Stronghold Extraction
5,Denver ➔ New York City,Sunday,19.5061,7.1611,10:00-12:59,38.6880,6.7054,Surge Capture
6,New York City ➔ Denver,Friday,16.7411,5.6296,10:00-12:59,38.1200,6.6363,Surge Capture
7,Boston ➔ South Florida,Thursday,15.6514,3.4431,10:00-12:59,32.2868,4.6444,Surge Capture
8,DMV ➔ Denver,Sunday,12.8511,9.2896,13:00-15:59,24.5787,6.2857,Surge Capture
9,New York City ➔ Dallas,Tuesday,15.7274,2.7778,07:00-09:59,12.4509,4.2105,Surge Capture


## Step 4 — Revenue Opportunity Quantification (SMID / Challenger)

### Fleet parameters (Q1 Challenger Fleet)

| Parameter | Low Scenario (10T) | High Scenario (17T) |
|---|---|---|
| Fleet hours (Q1) | 2,600 hrs | 4,420 hrs |
| Active tails | 10 | 17 |
| Hours per tail (Q1) | 260 hrs | 260 hrs |
| Hourly rate | $11,000 | $11,000 |
| Corridor allocation | 20 corridors | 20 corridors |

---

### SLI — Surge Loyalty Index (SMID)

> **SLI = `tod_wup_share_pct` / `NATIONAL_BASELINE`**
> where `NATIONAL_BASELINE = 3.4%` (overall WUP SMID long-haul share, verified from reference notebook Cell 5:
> `overall_wup_share = (total_wup_missions / total_long_haul) * 100`)

| SLI value | Interpretation |
|---|---|
| < 1.0 | WUP underweight vs SMID market average |
| 1.0 | WUP exactly at SMID market average |
| 1.5 | WUP 50% above market average — moderate signal |
| 2.5+ | WUP more than 2.5× market average — Stronghold threshold |

Example: South Florida → Denver at 10:00–12:59 has wup = 9.23% → SLI = 9.23 / 3.4 = **2.71** (Stronghold, +30% toggle)

---

### Hours-in-Window Formula

> **`hours = (Fleet_hrs / N_corridors) × (tod_velocity_pct / 100) × SLI`**

| Factor | Low | High | Meaning |
|---|---|---|---|
| `Fleet_hrs / N_corridors` | 130 hrs | 221 hrs | Per-corridor fleet allocation |
| `× tod_velocity_pct / 100` | varies | varies | Fraction in the peak TOD window |
| `× SLI` | varies | varies | WUP concentration multiplier |

---

### Variable Toggle — Pricing Uplift within Category

| Category | Condition for high toggle | High | Low |
|---|---|---|---|
| Stronghold Extraction | SLI > 2.5 | +30% | +25% |
| Surge Capture | — | +20% | +20% |
| Niche Protection | wup > 3.3% | +15% | +10% |
| Velocity Opportunity | velocity > 33% | +20% | +15% |

---

### Value Formula

> **`value = hours_in_window × hourly_rate × toggle_pct`**

Incremental revenue from applying the pricing uplift during the peak extraction window.
The SLI multiplier ensures opportunity sizing is proportional to WUP's actual presence —
high-SLI corridors capture more revenue-eligible hours from the same fleet allocation.


In [6]:
# -----------------------------------------------------------------------------------------
# CELL 6: VALUE CREATION TABLE (SMID / CHALLENGER FLEET)
#
# Challenger Base (Q1): Low  = 2,600 hrs (10 tails x 260 hrs)
#                        High = 4,420 hrs (17 tails x 260 hrs)
# Hourly rate          : $11,000
# SLI                  : tod_wup_share_pct / NATIONAL_BASELINE
#                         NATIONAL_BASELINE = 3.4% (overall WUP SMID share, ref Cell 5 source nb)
# Hours in window      : (Fleet hrs / N corridors) x tod_velocity_pct x SLI
#
# Toggle (variable within category, driven by SLI / wup / velocity):
#   Stronghold  SLI > 2.5  -> +30%   else +25%
#   Surge                  -> +20%
#   Niche       wup > 3.3% -> +15%   else +10%
#   Vel Opp     vel > 33%  -> +20%   else +15%
# -----------------------------------------------------------------------------------------

FLEET_LOW         = 2_600
FLEET_HIGH        = 4_420
N_CORRIDORS       = 20
HOURLY_RATE       = 11_000
NATIONAL_BASELINE = 3.4

def _get_toggle(cat, wup, vel):
    sli = wup / NATIONAL_BASELINE
    if   cat == "Stronghold Extraction": return 0.30 if sli  >  2.5 else 0.25
    elif cat == "Surge Capture":         return 0.20
    elif cat == "Niche Protection":      return 0.15 if wup  >  3.3 else 0.10
    elif cat == "Velocity Opportunity":  return 0.20 if vel  > 33.0 else 0.15
    else:                                return 0.00

df_value = df_categorized.copy()
df_value["sli"]        = (df_value["tod_wup_share_pct"] / NATIONAL_BASELINE).round(2)
df_value["toggle_pct"] = df_value.apply(
    lambda r: _get_toggle(str(r["category"]), r["tod_wup_share_pct"], r["tod_velocity_pct"]),
    axis=1
)

df_value["hours_low"]  = ((FLEET_LOW  / N_CORRIDORS) * (df_value["tod_velocity_pct"] / 100) * df_value["sli"]).round(1)
df_value["hours_high"] = ((FLEET_HIGH / N_CORRIDORS) * (df_value["tod_velocity_pct"] / 100) * df_value["sli"]).round(1)

df_value["value_low"]  = (df_value["hours_low"]  * HOURLY_RATE * df_value["toggle_pct"]).round(0).astype(int)
df_value["value_high"] = (df_value["hours_high"] * HOURLY_RATE * df_value["toggle_pct"]).round(0).astype(int)

df_value = df_value[[
    "corridor", "category",
    "tod_dimension", "tod_velocity_pct", "tod_wup_share_pct",
    "sli", "toggle_pct",
    "hours_low", "hours_high",
    "value_low", "value_high",
]]

summary = (
    df_value.groupby("category", observed=True)[["value_low", "value_high"]]
    .sum()
    .assign(
        value_low  = lambda d: d["value_low"].map("${:,.0f}".format),
        value_high = lambda d: d["value_high"].map("${:,.0f}".format),
    )
)
print("VALUE BY CATEGORY (Q1 Challenger Fleet)")
print("-" * 44)
display(summary)
print()

df_value["value_low"]  = df_value["value_low"].map("${:,.0f}".format)
df_value["value_high"] = df_value["value_high"].map("${:,.0f}".format)
display(df_value)


VALUE BY CATEGORY (Q1 Challenger Fleet)
--------------------------------------------


,value_low,value_high
category,,
Stronghold Extraction,"$1,448,150","$2,460,975"
Surge Capture,"$731,280","$1,243,220"
Niche Protection,"$214,445","$364,540"
No Signal,$0,$0


,corridor,category,tod_dimension,tod_velocity_pct,tod_wup_share_pct,sli,toggle_pct,hours_low,hours_high,value_low,value_high
0,South Florida ➔ Denver,Stronghold Extraction,10:00-12:59,42.5354,9.2284,2.71,0.30,149.9,254.7,"$494,670","$840,510"
1,Denver ➔ South Florida,Stronghold Extraction,10:00-12:59,41.3888,8.3518,2.46,0.25,132.4,225.0,"$364,100","$618,750"
2,Atlanta ➔ Denver,Stronghold Extraction,13:00-15:59,28.0420,7.9800,2.35,0.25,85.7,145.6,"$235,675","$400,400"
3,Boston ➔ Denver,Stronghold Extraction,07:00-09:59,23.8161,7.7810,2.29,0.25,70.9,120.5,"$194,975","$331,375"
4,Denver ➔ Boston,Stronghold Extraction,16:00-18:59,14.6359,8.6124,2.53,0.30,48.1,81.8,"$158,730","$269,940"
5,Denver ➔ New York City,Surge Capture,10:00-12:59,38.6880,6.7054,1.97,0.20,99.1,168.4,"$218,020","$370,480"
6,New York City ➔ Denver,Surge Capture,10:00-12:59,38.1200,6.6363,1.95,0.20,96.6,164.3,"$212,520","$361,460"
7,Boston ➔ South Florida,Surge Capture,10:00-12:59,32.2868,4.6444,1.37,0.20,57.5,97.8,"$126,500","$215,160"
8,DMV ➔ Denver,Surge Capture,13:00-15:59,24.5787,6.2857,1.85,0.20,59.1,100.5,"$130,020","$221,100"
9,New York City ➔ Dallas,Surge Capture,07:00-09:59,12.4509,4.2105,1.24,0.20,20.1,34.1,"$44,220","$75,020"


In [7]:
# -----------------------------------------------------------------------------------------
# CELL 7: ACTIVE CORRIDORS (EXCLUDE NO SIGNAL)
# Output: df_signals — corridors with an actionable premium category.
# -----------------------------------------------------------------------------------------

df_signals = (
    df_value[df_value["category"].astype(str) != "No Signal"]
    .reset_index(drop=True)
)

print(f"Active corridors : {len(df_signals)} / {len(df_value)}")
display(df_signals)


Active corridors : 15 / 20


,corridor,category,tod_dimension,tod_velocity_pct,tod_wup_share_pct,sli,toggle_pct,hours_low,hours_high,value_low,value_high
0,South Florida ➔ Denver,Stronghold Extraction,10:00-12:59,42.5354,9.2284,2.71,0.30,149.9,254.7,"$494,670","$840,510"
1,Denver ➔ South Florida,Stronghold Extraction,10:00-12:59,41.3888,8.3518,2.46,0.25,132.4,225.0,"$364,100","$618,750"
2,Atlanta ➔ Denver,Stronghold Extraction,13:00-15:59,28.0420,7.9800,2.35,0.25,85.7,145.6,"$235,675","$400,400"
3,Boston ➔ Denver,Stronghold Extraction,07:00-09:59,23.8161,7.7810,2.29,0.25,70.9,120.5,"$194,975","$331,375"
4,Denver ➔ Boston,Stronghold Extraction,16:00-18:59,14.6359,8.6124,2.53,0.30,48.1,81.8,"$158,730","$269,940"
5,Denver ➔ New York City,Surge Capture,10:00-12:59,38.6880,6.7054,1.97,0.20,99.1,168.4,"$218,020","$370,480"
6,New York City ➔ Denver,Surge Capture,10:00-12:59,38.1200,6.6363,1.95,0.20,96.6,164.3,"$212,520","$361,460"
7,Boston ➔ South Florida,Surge Capture,10:00-12:59,32.2868,4.6444,1.37,0.20,57.5,97.8,"$126,500","$215,160"
8,DMV ➔ Denver,Surge Capture,13:00-15:59,24.5787,6.2857,1.85,0.20,59.1,100.5,"$130,020","$221,100"
9,New York City ➔ Dallas,Surge Capture,07:00-09:59,12.4509,4.2105,1.24,0.20,20.1,34.1,"$44,220","$75,020"


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8: PREMIUM EXTRACTION — FORMATTED CATEGORY TABLES (SMID / CHALLENGER)
# ─────────────────────────────────────────────────────────────────────────────

from IPython.display import HTML

abbrev_map = {
    'South Florida':  'S. Fla',
    'New York City':  'NYC',
    'Minneapolis':    'Mpls',
    'Atlanta':        'ATL',
    'Boston':         'BOS',
    'Chicago':        'ORD',
    'Dallas':         'DAL',
    'Denver':         'DEN',
    'DMV':            'DMV',
}

def shorten(corridor):
    parts = corridor.split(' ➔ ')
    return ' → '.join(abbrev_map.get(p, p) for p in parts)

category_meta = {
    'Stronghold Extraction': 'HIGH-CONVICTION CORRIDOR PAIRS',
    'Surge Capture':         'MOMENTUM CORRIDOR PAIRS',
    'Niche Protection':      'LOWER-INTENSITY CORRIDOR PAIRS',
    'Velocity Opportunity':  'HIGH-DEMAND UNDERWEIGHT PAIRS',
}

def _get_toggle(cat, wup, vel):
    sli = wup / 3.4
    if   cat == 'Stronghold Extraction': return 0.30 if sli  >  2.5 else 0.25
    elif cat == 'Surge Capture':         return 0.20
    elif cat == 'Niche Protection':      return 0.15 if wup  >  3.3 else 0.10
    elif cat == 'Velocity Opportunity':  return 0.20 if vel  > 33.0 else 0.15
    else:                                return 0.00

_df = df_categorized.copy()
_df['toggle_pct'] = _df.apply(
    lambda r: _get_toggle(str(r['category']), r['tod_wup_share_pct'], r['tod_velocity_pct']),
    axis=1
)
_df['sli']      = _df['tod_wup_share_pct'] / 3.4
_df['val_low']  = (2600 / 20) * (_df['tod_velocity_pct'] / 100) * _df['sli'] * 11000 * _df['toggle_pct']
_df['val_high'] = (4420 / 20) * (_df['tod_velocity_pct'] / 100) * _df['sli'] * 11000 * _df['toggle_pct']
_active = _df[_df['category'].astype(str) != 'No Signal'].reset_index(drop=True)

CSS = """
<style>
.pex-wrap { font-family: 'Segoe UI', Arial, sans-serif; max-width: 860px; margin-bottom: 40px; }
.pex-meta { font-size: 11px; font-weight: 700; letter-spacing: 1.5px; color: #0055cc; margin-bottom: 6px; }
.pex-title { font-size: 26px; font-weight: 800; color: #0d1b3e;
             border-left: 5px solid #0055cc; padding-left: 12px; margin: 0 0 14px 0; }
.pex-table { width: 100%; border-collapse: collapse; font-size: 13px; }
.pex-table thead tr { background: #0d1b3e; color: #fff; }
.pex-table thead th { padding: 9px 12px; text-align: left; font-weight: 600; }
.pex-table thead th.right { text-align: right; }
.pex-table tbody tr { border-bottom: 1px solid #e8e8e8; }
.pex-table tbody tr:hover { background: #f5f8ff; }
.pex-table td { padding: 10px 12px; vertical-align: top; }
.pex-table td.num { color: #0055cc; font-weight: 700; width: 32px; }
.pex-table td.right { text-align: right; font-variant-numeric: tabular-nums; }
.pex-table td.toggle-hi  { color: #1a7a1a; font-weight: 700; }
.pex-table td.toggle-lo  { color: #b35c00; font-weight: 700; }
.pex-table tfoot tr { background: #f0f0f0; font-weight: 700; }
.pex-table tfoot td { padding: 9px 12px; }
.pex-table tfoot td.right { text-align: right; }
.pex-spacer { height: 36px; }
</style>
"""

hi_thresh = {
    'Stronghold Extraction': 0.30,
    'Surge Capture':         0.20,
    'Niche Protection':      0.15,
    'Velocity Opportunity':  0.20,
}

def build_table(cat_name, df_cat, row_start):
    subtitle = category_meta[cat_name]
    ht = hi_thresh[cat_name]
    sub_low  = df_cat['val_low'].sum()
    sub_high = df_cat['val_high'].sum()

    rows_html = ''
    for i, (_, r) in enumerate(df_cat.iterrows()):
        tc = 'toggle-hi' if r['toggle_pct'] >= ht else 'toggle-lo'
        rows_html += f"""
        <tr>
          <td class="num">{row_start + i}</td>
          <td>{shorten(r['corridor'])}</td>
          <td>
            <div style="font-weight:700;color:#0d1b3e">{r['dow_dimension']}</div>
            <div style="font-size:11px;color:#666">{r['tod_dimension']}</div>
          </td>
          <td>{r['tod_wup_share_pct']:.1f}%</td>
          <td class="{tc}">+{int(round(r['toggle_pct']*100))}%</td>
          <td class="right">${r['val_low']:,.0f}</td>
          <td class="right">${r['val_high']:,.0f}</td>
        </tr>"""

    return f"""
<div class="pex-wrap">
  <div class="pex-meta">CHALLENGER &nbsp;|&nbsp; {cat_name.upper()} &nbsp;|&nbsp; {subtitle}</div>
  <h2 class="pex-title">{cat_name} Toggles</h2>
  <table class="pex-table">
    <thead>
      <tr>
        <th>#</th><th>Corridor / Pair</th><th>Operational Detail</th>
        <th>Intensity</th><th>Toggle</th>
        <th class="right">Q1 Low (10T)</th><th class="right">Q1 High (17T)</th>
      </tr>
    </thead>
    <tbody>{rows_html}</tbody>
    <tfoot>
      <tr>
        <td colspan="5">SUBTOTAL</td>
        <td class="right">${sub_low:,.0f}</td>
        <td class="right">${sub_high:,.0f}</td>
      </tr>
    </tfoot>
  </table>
</div>
<div class="pex-spacer"></div>"""

html_out = CSS
row_num = 1
for cat in ['Stronghold Extraction', 'Surge Capture', 'Niche Protection', 'Velocity Opportunity']:
    df_cat = _active[_active['category'].astype(str) == cat]
    if df_cat.empty:
        continue
    html_out += build_table(cat, df_cat, row_num)
    row_num += len(df_cat)

grand_low  = _active['val_low'].sum()
grand_high = _active['val_high'].sum()
html_out += f"""
<div class="pex-wrap">
  <table class="pex-table">
    <tfoot>
      <tr>
        <td colspan="5" style="font-size:15px;">GRAND TOTAL (Q1 Challenger Fleet)</td>
        <td class="right" style="font-size:15px;">${grand_low:,.0f}</td>
        <td class="right" style="font-size:15px;">${grand_high:,.0f}</td>
      </tr>
    </tfoot>
  </table>
</div>"""

display(HTML(html_out))

---

## Results Interpretation Guide

### Priority order for revenue management action

| Priority | Category | Corridors | Action |
|---|---|---|---|
| 1 | **Stronghold Extraction** | 5 | Maximum toggles immediately — Denver-hub WUP dominance is clear |
| 2 | **Surge Capture** | 5 | Deploy within the week — strong WUP + active demand |
| 3 | **Niche Protection** | 5 | Schedule for next cycle — moderate signal, protect before competition moves |
| 4 | **No Signal** | 5 | Monitor only — insufficient dual-criterion evidence |

### Key SMID finding

The SMID set is heavily **Denver-hub concentrated** — 8 of the 10 Stronghold + Surge corridors
involve Denver as an origin or destination. This reflects WUP's outsized concentration in Denver-routed
SMID traffic. Revenue management should treat Denver-hub windows as a **franchise asset** to defend.

### Low vs High scenario

Low = 10 tails × 260 hrs = 2,600 Q1 hours. High = 17 tails × 260 hrs = 4,420 Q1 hours.
Value scales linearly — the toggle recommendations and corridor priorities are identical in both scenarios.
